# Reproducing Chang et al. (2023) from dandiset 000232

**Paper.** Chang Y-T, Finkel EA, Xu D, O'Connor DH. Rule-based modulation of a sensorimotor
transformation across cortical areas. bioRxiv, [doi:10.1101/2023.08.21.554194](https://doi.org/10.1101/2023.08.21.554194).
This attempt used the full text of preprint version 2 (2024-03-09, CC-BY), as served by eLife for
its reviewed preprint 92620v2, whose source is bioRxiv version 2; bioRxiv refused the download. Table 1
and the table of coding-dimension trial types are images in that copy and were not read.

**Dandiset.** [DANDI:000232](https://dandiarchive.org/dandiset/000232/0.240510.2038), version
0.240510.2038. The preprint cites no DANDI version (its data statement says "available upon request");
the paper's code repository cites this version's DOI, and it is the only published version. The
draft holds the same 179 files (same assets and blobs) and differs only in contributor metadata.

**What the dandiset holds.** 49 silicon-probe sessions of the cross-modal selection task (S1, S2, MM,
ALM and AMM, one area per session; trials and spike times of the units kept after the paper's quality
filter) and 141 optogenetic behaviour sessions (cross-modal task and tactile detection task).

**Code read.** The paper's analysis code, https://github.com/YitingChang/cross_modal_task at commit
227631ee133709df979b9914c9031cfa992c3736 (MATLAB; preprocessing, data arrays, one folder per figure).
Where the Methods leave a choice open, the analysis follows the code; where they disagree, the Methods'
choice is the main result and the code's a variant. Nothing from outside the dandiset is used as data.

| Claim | Tested |
|---|---|
| fig1e-recording-counts | tested |
| fig6-session-counts | tested |
| fig1d-performance | tested |
| fig1d-outcome-fractions | tested |
| fig1h-thit-tcr-selective | tested |
| fig2c-rule-prestim-units | tested |
| fig2d-stimulus-prestim-units | tested |
| fig2e-stimulus-poststim-units | tested |
| fig2f-prestim-predicts-touch-response | tested |
| fig3b-rule-decoding | tested |
| fig3c-rule-decoding-motor-stronger | tested |
| fig3de-stimulus-decoding-controls | tested |
| figs3-svm-rf-decoding | tested |
| fig4c-prestim-distance-predicts-divergence | tested |
| fig4e-subspace-overlap | tested |
| fig4f-prestim-overlap-predicts-poststim | tested |
| figs4ef-coding-dimensions | tested |
| figs4h-choice-right-left | tested |
| fig5b-first-hit-switches | tested |
| fig5de-motor-states-track-switch | tested |
| figs5-sensory-states-no-tracking | tested |
| fig6c-pre-inhibition-tactile | tested |
| figs6b-pre-inhibition-hit-rate | tested |
| fig6d-post-inhibition-tactile | tested |
| figs7gh-visual-sensitivity | tested |
| fig6-ppc-inhibition | tested |
| fig6fg-detection-task | tested |
| fig6ij-laser-only-licks | tested |
| fig1e-probe-locations | blocked (data missing: no histology or electrode coordinates) |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC

import xmodal as xm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 200)
warnings.filterwarnings("ignore", category=RuntimeWarning)
BLUE, GRAY, ORANGE = "#2a78d6", "#8a8984", "#eb6834"

started = time.time()
assets = xm.asset_urls()
files = xm.load_all(assets, spikes=True)
ephys = [f for f in files if f["kind"] == "ephys"]
opto = [f for f in files if f["kind"] == "opto"]
print(f"dandiset {xm.DANDISET_ID} version {xm.DANDISET_VERSION}: {len(assets)} NWB files, "
      f"{len(ephys)} with units, {len(opto)} without; loaded in {time.time() - started:.0f} s")

## Data check

Every check of the description against the values is stated under its output.

In [ ]:
info = pd.DataFrame([{"path": f["path"], "subject": f["subject"], "genotype": f["genotype"], "sex": f["sex"],
                      "kind": f["kind"], "description": f["description"], "trials": len(f["trials"]),
                      "units": f.get("n_units", 0)} for f in files])
print("Files by experiment description:")
print(info.groupby("description").agg(files=("path", "size"), mice=("subject", "nunique")).to_string())
print("\nSubjects:")
print(info.groupby("subject").agg(genotype=("genotype", "first"), sex=("sex", "first"),
                                  files=("path", "size"), ephys=("kind", lambda k: (k == "ephys").sum())).to_string())
print("\nArchive summary: 18 subjects, 179 files; files hold", info.subject.nunique(), "subjects,", len(info), "files")

The descriptions name one recorded area per ephys file and one inhibited area (or sham) and one task
per optogenetic file. The 18 subjects and 179 files agree with the archive's summary.

In [ ]:
rows = []
for f in ephys:
    rows.append({"session": f["path"].split("/")[1][:31], "subject": f["subject"], "area": f["area"],
                 "group_location": f["group_location"], "electrode_locations": ",".join(f["electrode_locations"]),
                 "units": f["n_units"], "unit_columns": ",".join(f["unit_columns"]), "trials": len(f["trials"])})
eph_info = pd.DataFrame(rows)
print("Area in the description equals the electrode group's location in every file:",
      bool((eph_info.area == eph_info.group_location).all()))
print("Electrode-level locations:", eph_info.electrode_locations.unique().tolist())
print("Units table columns:", eph_info.unit_columns.unique().tolist())
print("\nSessions, mice and units per area:")
print(eph_info.groupby("area").agg(sessions=("session", "size"), mice=("subject", "nunique"),
                                   units=("units", "sum"), units_min=("units", "min"), units_max=("units", "max"),
                                   trials_min=("trials", "min"), trials_max=("trials", "max")).to_string())
f0 = ephys[0]
print("\nTrials table columns (ephys):", f0["trial_columns"])
import h5py, remfile
with h5py.File(remfile.File(assets.set_index("path").loc[f0["path"], "url"]), "r") as h:
    e = h["general/extracellular_ephys/electrodes"]
    print("Electrode x, y, z all NaN in", f0["path"], ":", bool(np.isnan(np.r_[e["x"][:], e["y"][:], e["z"][:]]).all()),
          "; unit ids:", h["units/id"][:5].tolist(), "...")

In [ ]:
# Trial coding: block, stimulus and response against the lick times.
rows = []
for f in ephys:
    t = xm.code_trials(f["trials"])
    first_side = []
    for r, l, s0 in zip(t.r_licks, t.l_licks, t.stim_on):
        rr, ll = r[(r - s0 > 0.1) & (r - s0 < 2)], l[(l - s0 > 0.1) & (l - s0 < 2)]
        first_side.append(3 if len(rr) + len(ll) == 0 else 1 if len(ll) == 0 or (len(rr) and rr[0] < ll[0]) else 2)
    first_side = np.array(first_side)
    lick_pre = xm.licks_in(t, -1, 0.1)
    gaps = t.start.to_numpy()[1:] - t.stop.to_numpy()[:-1]
    rows.append({"session": f["path"].split("/")[1][:31], "trials": len(t), "cue_trials": int((t.b == 0).sum()),
                 "no_stimulus": int((t.s == 0).sum()), "response_values": ",".join(map(str, sorted(t.response.unique()))),
                 "response_eq_first_lick_side": round(float(np.mean(first_side[t.d > 0] == t.d[t.d > 0])), 3),
                 "trials_lick_-1_to_0.1s": int(lick_pre.sum()),
                 "stim_minus_start_min_s": round(float((t.stim_on - t.start).min()), 3),
                 "stop_minus_stim_min_s": round(float((t.stop - t.stim_on).min()), 3),
                 "gaps_over_10ms": int((gaps > 0.01).sum()), "gap_total_s": round(float(gaps[gaps > 0.01].sum()), 0)})
coding = pd.DataFrame(rows)
print(coding.describe().T[["min", "50%", "max"]].to_string())
allt = pd.concat([xm.code_trials(f["trials"]).assign(area=f["area"]) for f in ephys], ignore_index=True)
print("\nBlock type x trial type x response, all ephys sessions:")
print(pd.crosstab([allt.block_type, allt.trial_type], allt.response).to_string())
print("\nStimulus types:", allt.vis_stim_type.unique().tolist(), allt.som_stim_type.unique().tolist())

`response` agrees with the side of the first lick between 0.1 and 2 s after the stimulus in at least the
minimum fraction printed (the remainder are licks the behavioural system timed differently); 1 is a right
lick, 2 a left lick and 0 no lick. "Change" rows are the cue trials (water given on the 9th trial of a
block), left out of all analyses as in the paper's code. No trial in any ephys file has a lick between -1 and 0.1 s of the stimulus: the trials the
Methods remove for pre-stimulus licking (about 25%) are already absent, and the gaps between trials mark
where they were. The dandiset does not say this. The stimulus comes at least 1 s after the trial start and
the trial ends at least 3.4 s after it, so every analysis window lies inside a trial.

In [ ]:
rows = []
for f in ephys:
    t = f["trials"]
    dur = float((t.stop - t.start).sum())
    rates = np.array([len(s) / dur for s in f["spikes"]])
    obs_is_trials = all(np.allclose(o, np.c_[t.start, t.stop], atol=2e-3) for o in f["obs_intervals"])
    starts, stops = t.start.to_numpy() - 2e-3, t.stop.to_numpy() + 2e-3
    in_trial = np.mean([np.mean(np.searchsorted(starts, sp, side="right") - 1 == np.searchsorted(stops, sp, side="left"))
                        for sp in f["spikes"]])
    rows.append({"session": f["path"].split("/")[1][:31], "area": f["area"], "units": f["n_units"],
                 "sorted": all(np.all(np.diff(s) >= 0) for s in f["spikes"]),
                 "obs_intervals_are_trials": obs_is_trials, "spikes_in_trials": round(float(in_trial), 4),
                 "rate_min_hz": rates.min().round(2), "rate_max_hz": rates.max().round(1),
                 "spike_hash": hash(np.concatenate(f["spikes"][:3]).tobytes())})
units_check = pd.DataFrame(rows)
print(units_check.agg({"sorted": "all", "obs_intervals_are_trials": "all", "spikes_in_trials": "min",
                       "rate_min_hz": "min", "rate_max_hz": "max", "units": "sum"}).to_string())
print("Sessions whose first three spike trains are identical to another session's:",
      int(units_check.spike_hash.duplicated().sum()))

Spike times are sorted, each unit's observation intervals are the trial intervals, and spikes are stored
only inside trials. No unit has a mean rate within trials above 50 Hz, consistent with the code's removal
of such units, and the unit counts per area equal the paper's after its quality filter (see
fig1e-recording-counts), so the Units tables hold the selected units only. They carry no quality metrics,
waveforms, channel or depth, and unit ids run from 0, so the units the paper names by cluster number
(for example unit 127 of YT084, 2020-02-01) cannot be found. Electrode coordinates are NaN and
electrode locations "unknown". No two sessions share spike trains, so no recording is deposited twice.

In [ ]:
# Clock check: decoding the stimulus (tactile against visual, correct trials) in 100 ms windows around its onset.
rng_clock = np.random.default_rng(101)
windows = [(-0.3, -0.2), (-0.2, -0.1), (-0.1, 0.0), (0.0, 0.1), (0.1, 0.2)]
rows = []
for f in ephys:
    t = xm.code_trials(f["trials"])
    sel, block, stim = xm.correct_trials(t)
    on = t.stim_on.to_numpy()[sel]
    acc = {}
    for lo, hi in windows:
        x = xm.binned_rates(f["spikes"], on, lo, hi, hi - lo)[:, :, 0]
        acc[f"{lo:+.1f}"] = xm.cv_accuracy(x, stim, rng_clock)
    x = xm.binned_rates(f["spikes"], on, 0.0, 0.1, 0.1)[:, :, 0]
    acc["shuffled_0.0"] = xm.cv_accuracy(x, rng_clock.permutation(stim), rng_clock)
    rows.append({"session": f["path"].split("/")[1][:31], "area": f["area"], "units": f["n_units"], **acc})
clock = pd.DataFrame(rows)
print(clock.groupby("area")[[f"{lo:+.1f}" for lo, _ in windows] + ["shuffled_0.0"]].median().round(3).to_string())
clock["jump"] = clock["+0.0"] - clock["-0.1"]
print("\nSessions where accuracy 0 to 100 ms exceeds -100 to 0 ms by at least 0.05:",
      int((clock.jump >= 0.05).sum()), "of", len(clock))
print(clock.loc[clock.jump < 0.05, ["session", "area", "units", "-0.1", "+0.0", "+0.1"]].round(3).to_string())

In 43 of 49 sessions the stimulus is decoded from 0 to 100 ms after the onset in the trial table, at a
median accuracy of 0.64 to 0.89 by area, and not from the 100 ms windows before it (medians 0.48 to
0.56, shuffled 0.50 to 0.56), which shows that spike times and trial times share a clock. In the six
sessions listed the rise in the first 100 ms is below 0.05. Three of them (EF0147 2019-02-06, YT085
2020-03-10 and 2020-03-22) rise in the next window, 100 to 200 ms (0.64 to 0.75); for the other three,
YT084 2020-02-02 and 2020-02-03 (AMM, 2 and 5 units) and YT086 2020-03-20 (MM, 5 units, 0.63 before and
0.65 after), the shared clock is not established by this check.

In [ ]:
# Optogenetic files: conditions, laser timing, trial labels and the code's performance check.
rows = []
for f in opto:
    t = f["trials"]
    rel = np.round((t.opto_on - t.stim_on).to_numpy(), 1)
    dur = np.round((t.opto_off - t.opto_on).to_numpy(), 1)
    licks = xm.opto_licks(t, f["task"])
    rows.append({"session": f["path"].split("/")[1][:31], "mouse": f["subject"], "genotype": f["genotype"],
                 "task": f["task"], "condition": f["condition"], "trials": len(t),
                 "laser_onsets": ",".join(sorted({str(v) for v in rel[~np.isnan(rel)]})),
                 "laser_durations": ",".join(sorted({str(v) for v in dur[~np.isnan(dur)]})),
                 "laser_trials": int((~np.isnan(rel)).sum()),
                 "start_eq_stim": bool(np.allclose(t.start, t.stim_on)),
                 "overlapping_trials": int((t.start.to_numpy()[1:] < t.stop.to_numpy()[:-1]).sum()),
                 "passes_code_check": xm.cross_modal_passed(licks) if f["task"] == "cross-modal" else None,
                 "licks": licks})
optos = pd.DataFrame(rows)
print(optos.groupby(["task", "condition"]).agg(sessions=("session", "size"), mice=("mouse", "nunique"),
                                              genotypes=("genotype", lambda g: ",".join(sorted(set(g)))),
                                              laser_trials_median=("laser_trials", "median"),
                                              passes_code_check=("passes_code_check", lambda p: int(p.sum()) if p.notna().all() else "not run")).to_string())
print("\nLaser onset relative to the reference time (s):", sorted(set(",".join(optos.laser_onsets).split(","))))
print("Laser durations (s):", sorted(set(",".join(optos.laser_durations).split(","))))
onsets = np.concatenate([np.round((f["trials"].opto_on - f["trials"].stim_on).to_numpy(), 1) for f in opto])
durs = np.concatenate([np.round((f["trials"].opto_off - f["trials"].opto_on).to_numpy(), 1) for f in opto])
print("Laser trials by onset:", pd.Series(onsets[~np.isnan(onsets)]).value_counts().to_dict(),
      "; by duration:", pd.Series(durs[~np.isnan(durs)]).value_counts().to_dict())
print("start_time equals stimulus onset in every file:", bool(optos.start_eq_stim.all()),
      "; files with overlapping trial intervals:", int((optos.overlapping_trials > 0).sum()))
ot = pd.concat([f["trials"].assign(task=f["task"]) for f in opto], ignore_index=True)
print("\nTrial types by task (rows) with somatosensory / visual onsets present:")
print(ot.groupby(["task", "trial_type"]).agg(trials=("stim_on", "size"), som=("som_on", lambda x: x.notna().sum()),
                                             vis=("vis_on", lambda x: x.notna().sum()),
                                             laser=("opto_on", lambda x: x.notna().sum())).to_string())
print("\nDuplicate trial tables among optogenetic files:",
      int(pd.Series([hash(f["trials"].stim_on.to_numpy().tobytes()) for f in opto]).duplicated().sum()))

Laser onsets fall at -0.8 s (pre-stimulus inhibition, 0.8 s long) or 0 s (from the stimulus, 2 s long)
from the reference time, as the Methods state; on laser-only ("catch") trials the reference time is the
laser offset for short trains and the onset for long ones. Two laser trials start at -1.6 and -2.7 s and
one lasts 9.5 s; as in the code, any onset before 0 counts as pre-stimulus. In these files `start_time` equals the
stimulus onset and trial intervals overlap, so `start_time` and `stop_time` do not delimit trials. In the
detection task, trials without a whisker stimulus are labelled "Stim_Vis" though no visual stimulus is
given (no visual onset). The code's performance check passes all cross-modal sessions but one sham
session, so the files hold the sessions the paper kept (the counts match, fig6-session-counts), and the
one sham session that fails here is kept as deposited.

In [ ]:
# Per-session arrays for the ephys analyses, and the code's session checks.
sessions = [xm.ephys_session(f) for f in ephys]
main = [s for s in sessions if s["area"] in xm.AREAS]
rows = []
for s in main:
    t = s["trials"]
    n = {cat: int(xm.is_category(t, key).sum()) for key, cat in xm.CATEGORIES.items()}
    rows.append({"session": s["path"].split("/")[1][:31], "area": s["area"], **n})
counts = pd.DataFrame(rows)
needed = [c for c in xm.CATEGORIES.values() if c not in ("TTV", "VVT")]
print("Sessions with fewer than 2 trials of a category the code requires:", int((counts[needed] < 2).any(axis=1).sum()))
print("Trials per category over the 40 sessions (min, median, max):")
print(counts[list(xm.CATEGORIES.values())].agg(["min", "median", "max"]).T.to_string())
print("\nRates used: 100 ms bins", main[0]["r100"].shape[2], "; smoothed 10 ms bins", main[0]["r10"].shape[2],
      "; max smoothed rate over all sessions", round(max(float(s["r10"].max()) for s in main), 1), "Hz")

## fig1e-recording-counts

> Single units were recorded in S1 (6 mice, 10 sessions, 177 neurons), S2 (5, 8, 162), MM (7, 9, 140) and
> ALM (8, 13, 256), one area per session, and in AMM (110 neurons).

In [ ]:
paper = pd.DataFrame({"mice": [6, 5, 7, 8, None], "sessions": [10, 8, 9, 13, None], "units": [177, 162, 140, 256, 110]},
                     index=["S1", "S2", "MM", "ALM", "AMM"])
found = eph_info.groupby("area").agg(mice=("subject", "nunique"), sessions=("session", "size"), units=("units", "sum"))
table = paper.join(found, lsuffix="_paper", rsuffix="_dandiset")
print(table.to_string())
print("All stated counts match:", bool(all((table[c + "_paper"].isna() | (table[c + "_paper"] == table[c + "_dandiset"])).all()
                                         for c in ("mice", "sessions", "units"))))

**Result: reproduced.** Sessions, mice and units per area match the paper exactly: S1 10 sessions, 6 mice, 177 units; S2 8, 5, 162; MM 9, 7, 140; ALM 13, 8, 256; AMM 110 units (9 sessions, 7 mice).

## fig6-session-counts

> Optogenetic sessions and mice per condition in the cross-modal and detection tasks (Figure 6 legend).

In [ ]:
paper6 = {("cross-modal", "S1/S2"): (4, 10), ("cross-modal", "MM"): (4, 11), ("cross-modal", "ALM"): (4, 10),
          ("cross-modal", "sham"): (7, 28), ("cross-modal", "AMM"): (7, 21), ("cross-modal", "PPC"): (7, 21),
          ("detection", "sham"): (5, 9), ("detection", "S1/S2"): (3, 3), ("detection", "MM"): (3, 5),
          ("detection", "ALM"): (3, 4), ("detection", "AMM"): (3, 4), ("detection", "PPC"): (3, 4)}
g = optos.groupby(["task", "condition"]).agg(mice=("mouse", "nunique"), sessions=("session", "size"))
rows = [{"task": k[0], "condition": k[1], "mice_paper": v[0], "mice": g.loc[k, "mice"],
         "sessions_paper": v[1], "sessions": g.loc[k, "sessions"]} for k, v in paper6.items()]
t6 = pd.DataFrame(rows)
print(t6.to_string(index=False))
print("Counts matching:", int((t6.mice == t6.mice_paper).sum() + (t6.sessions == t6.sessions_paper).sum()), "of", 2 * len(t6))

**Result: reproduced.** All 24 counts of optogenetic sessions and mice per condition match the Figure 6 legend.

## fig1d-performance and fig1d-outcome-fractions

> Performance was about 75% correct under both rules (touch 74 ± 1%, light 75 ± 1%, 12 mice), with no
> difference (paired t-test, p = 0.56). Outcome fractions, touch against light: hit 36/35%, correct
> rejection 38/40%, miss 13/14%, false alarm 13/12%.

Reading (a) uses the trials as deposited, reading (b) drops the last 20 trials of each session (Methods).

In [ ]:
def block_outcomes(t):
    c = t.category[t.category != ""]
    out = {}
    for blk, hit, cr, miss, fa in (("touch", "TTT", "TVN", "TTN", ["TTV", "TVT", "TVV"]),
                                   ("light", "VVV", "VTN", "VVN", ["VTT", "VTV", "VVT"])):
        n = c.str[0].eq("T" if blk == "touch" else "V").sum()
        out.update({f"{blk}_hit": (c == hit).sum() / n, f"{blk}_cr": (c == cr).sum() / n,
                    f"{blk}_miss": (c == miss).sum() / n, f"{blk}_fa": c.isin(fa).sum() / n})
        out[f"{blk}_perf"] = out[f"{blk}_hit"] + out[f"{blk}_cr"]
    return out

perf = pd.DataFrame([{"subject": s["subject"], "area": s["area"], "reading": r,
                      **block_outcomes(s["trials"] if r == "a" else s["trials"].iloc[:-20])}
                     for s in main for r in ("a", "b")])
fig1d = {}
for r, g in perf.groupby("reading"):
    m = g.groupby("subject").mean(numeric_only=True)
    test = stats.ttest_rel(m.touch_perf, m.light_perf)
    fig1d[r] = (m, test)
    print(f"Reading ({r}): {len(m)} mice; percent, mean and SEM over mice")
    print(pd.DataFrame({"mean": 100 * m.mean(), "sem": 100 * m.sem(), "min_mouse": 100 * m.min(),
                        "max_mouse": 100 * m.max()}).round(1).T.to_string())
    print(f"  paired t-test touch against light performance: t = {test.statistic:.3f}, p = {test.pvalue:.3f}\n")

In [ ]:
expected = {"touch_perf": (74, 1), "light_perf": (75, 1), "touch_hit": (36, 1), "light_hit": (35, 2),
            "touch_cr": (38, 1), "light_cr": (40, 1), "touch_miss": (13, 1), "light_miss": (14, 1),
            "touch_fa": (13, 1), "light_fa": (12, 1)}
for r, (m, test) in fig1d.items():
    within = {k: abs(100 * m[k].mean() - v) <= tol for k, (v, tol) in expected.items()}
    print(f"Reading ({r}) within tolerance:", within)
    print(f"  fractions within tolerance: {sum(within[k] for k in expected if 'perf' not in k)} of 8")
print("\nPer area (reading a, mean over sessions):")
print((100 * perf[perf.reading == "a"].groupby("area")[["touch_perf", "light_perf"]].mean()).round(1).to_string())

**Result: reproduced.** Under reading (a), the trials as deposited, performance is 73.6 ± 0.8% in respond-to-touch and 74.8 ± 1.5% in respond-to-light blocks (mean and SEM over 12 mice), both within 1 point of the paper, and the paired t-test gives p = 0.561 (paper 0.56). Reading (b), without the last 20 trials, gives 72.8 and 74.2% (p = 0.558); its touch value is outside tolerance.

**Result: reproduced.** Under reading (a) all eight outcome fractions are within the paper's SEM (touch against light: hit 35.6 against 35.2%, correct rejection 38.1 against 39.6%, miss 13.0 against 13.6%, false alarm 13.3 against 11.6%); reading (b) puts 6 of 8 within.

## Single-neuron discrimination (fig1h, fig2c, fig2d, fig2e, fig2f)

> 18 to 33% of neurons discriminate tactile hits (tHit) from tactile correct rejections (tCR) in the first
> 150 ms (S1 28.8, S2 17.9, MM 32.9, ALM 23.4%), mostly with AUC above 0.5 (fig1h). Before the stimulus,
> S1 4.5, S2 2.5, MM 21.4, ALM 10.2% discriminate the rule (fig2c); almost none discriminate the stimulus
> (fig2d; 0, 0, 0, 0.8%); after it, 37.3, 35.2, 15, 14.1% do (fig2e). Among rule-discriminating neurons, the
> pre-stimulus rule preference predicts the early touch-response modulation (fig2f; MM r = 0.93, 30
> neurons; ALM r = 0.83, 26 neurons).

The bootstrap resamples are shared by all units and bins of a session (the code draws them separately);
each unit's interval depends only on its own resamples.

In [ ]:
rng_roc = np.random.default_rng(202)
N_BOOT = 1000
roc = []
for s in main:
    t = s["trials"]
    hit, cr, vh, vcr = (xm.is_category(t, k) for k in (xm.T_HIT, xm.T_CR, xm.V_HIT, xm.V_CR))
    n = s["n_units"]
    b1h = xm.bootstrap_auc(s["r10"][hit][:, :, 10:25].reshape(hit.sum(), -1),
                           s["r10"][cr][:, :, 10:25].reshape(cr.sum(), -1), N_BOOT, rng_roc).reshape(N_BOOT, n, 15)
    rule = xm.bootstrap_auc(s["r100"][hit | vcr][:, :, 0], s["r100"][cr | vh][:, :, 0], N_BOOT, rng_roc)
    stim_pre = xm.bootstrap_auc(s["r100"][hit | cr][:, :, 0], s["r100"][vh | vcr][:, :, 0], N_BOOT, rng_roc)
    stim_post = xm.bootstrap_auc(s["r100"][hit | cr][:, :, 1], s["r100"][vh | vcr][:, :, 1], N_BOOT, rng_roc)
    roc.append({"area": s["area"], "subject": s["subject"], "n": n, "trials": (hit.sum(), cr.sum(), vh.sum(), vcr.sum()),
                "b1h": b1h, "rule": rule, "stim_pre": stim_pre, "stim_post": stim_post})

units = []
for area in xm.AREAS:
    R = [r for r in roc if r["area"] == area]
    n_area = sum(r["n"] for r in R)
    b1h = np.concatenate([r["b1h"] for r in R], axis=1)
    sig1h = xm.bonferroni_significant(b1h.reshape(N_BOOT, -1), n_area).reshape(n_area, 15)
    df = pd.DataFrame({"area": area, "subject": np.concatenate([[r["subject"]] * r["n"] for r in R]),
                       "selective": xm.consecutive(sig1h, 3), "selective_2bins": xm.consecutive(sig1h, 2),
                       "selective_4bins": xm.consecutive(sig1h, 4),
                       "auc_0_150": b1h.mean(axis=(0, 2)), "auc_0_50": b1h[:, :, :5].mean(axis=(0, 2))})
    for name in ("rule", "stim_pre", "stim_post"):
        b = np.concatenate([r[name] for r in R], axis=1)
        df[name + "_sig"] = xm.bonferroni_significant(b, n_area)
        df[name + "_sig_uncorrected"] = xm.bonferroni_significant(b, 1)
        df[name + "_auc"] = b.mean(axis=0)
    units.append(df)
units = pd.concat(units, ignore_index=True)
print("Trials per class (tHit, tCR, vHit, vCR), min over sessions:", np.min([r["trials"] for r in roc], axis=0))

In [ ]:
summary = units.groupby("area", sort=False).agg(
    neurons=("selective", "size"), fig1h_pct=("selective", "mean"), fig2c_pct=("rule_sig", "mean"),
    fig2d_pct=("stim_pre_sig", "mean"), fig2e_pct=("stim_post_sig", "mean"))
for c in summary.columns[1:]:
    summary[c] = (100 * summary[c]).round(1)
summary["fig1h_paper"] = [28.8, 17.9, 32.9, 23.4]
summary["fig2c_paper"] = [4.5, 2.5, 21.4, 10.2]
summary["fig2d_paper"] = [0, 0, 0, 0.8]
summary["fig2e_paper"] = [37.3, 35.2, 15.0, 14.1]
summary["fig2c_neurons"] = units.groupby("area", sort=False).rule_sig.sum()
print(summary.to_string())
sel = units[units.selective]
print(f"\nfig1h: selective neurons with mean AUC above 0.5: {(sel.auc_0_150 > 0.5).sum()} of {len(sel)} "
      f"({100 * (sel.auc_0_150 > 0.5).mean():.1f}%); by area:",
      sel.groupby("area", sort=False).auc_0_150.apply(lambda a: round(100 * (a > 0.5).mean(), 1)).to_dict())
print("\nVariants: percent selective with 2 or 4 consecutive bins, and fig2 percentages without Bonferroni correction")
print((100 * units.groupby("area", sort=False)[["selective_2bins", "selective_4bins", "rule_sig_uncorrected",
                                                 "stim_pre_sig_uncorrected", "stim_post_sig_uncorrected"]].mean()).round(1).to_string())
print("\nBy mouse (percent; neurons):")
print(units.groupby(["area", "subject"], sort=False).agg(n=("selective", "size"), fig1h=("selective", "mean"),
                                                         fig2c=("rule_sig", "mean"), fig2e=("stim_post_sig", "mean"))
      .assign(fig1h=lambda d: (100 * d.fig1h).round(0), fig2c=lambda d: (100 * d.fig2c).round(0),
              fig2e=lambda d: (100 * d.fig2e).round(0)).to_string())

In [ ]:
# How much the percentages move with the bootstrap draws: the same analysis with four other seeds.
reps = []
for seed in (11, 12, 13, 14):
    rng_rep = np.random.default_rng(seed)
    per_area = {a: {"b1h": [], "rule": [], "stim_pre": [], "stim_post": []} for a in xm.AREAS}
    for s in main:
        t = s["trials"]
        hit, cr, vh, vcr = (xm.is_category(t, k) for k in (xm.T_HIT, xm.T_CR, xm.V_HIT, xm.V_CR))
        n = s["n_units"]
        d = per_area[s["area"]]
        d["b1h"].append(xm.bootstrap_auc(s["r10"][hit][:, :, 10:25].reshape(hit.sum(), -1),
                                         s["r10"][cr][:, :, 10:25].reshape(cr.sum(), -1), N_BOOT, rng_rep).reshape(N_BOOT, n, 15))
        d["rule"].append(xm.bootstrap_auc(s["r100"][hit | vcr][:, :, 0], s["r100"][cr | vh][:, :, 0], N_BOOT, rng_rep))
        d["stim_pre"].append(xm.bootstrap_auc(s["r100"][hit | cr][:, :, 0], s["r100"][vh | vcr][:, :, 0], N_BOOT, rng_rep))
        d["stim_post"].append(xm.bootstrap_auc(s["r100"][hit | cr][:, :, 1], s["r100"][vh | vcr][:, :, 1], N_BOOT, rng_rep))
    for a, d in per_area.items():
        b1h = np.concatenate(d["b1h"], axis=1)
        na = b1h.shape[1]
        row = {"seed": seed, "area": a,
               "fig1h": 100 * xm.consecutive(xm.bonferroni_significant(b1h.reshape(N_BOOT, -1), na).reshape(na, 15)).mean()}
        for k, name in (("rule", "fig2c"), ("stim_pre", "fig2d"), ("stim_post", "fig2e")):
            row[name] = 100 * xm.bonferroni_significant(np.concatenate(d[k], axis=1), na).mean()
        reps.append(row)
reps = pd.DataFrame(reps)
print("Percent of neurons over four other seeds (min to max):")
print(reps.groupby("area", sort=False).agg(["min", "max"]).drop(columns="seed").round(1).to_string())

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(12, 2.6), sharey=True)
for ax, area in zip(axes, xm.AREAS):
    u = units[units.area == area]
    ax.hist(u.auc_0_150, bins=np.arange(0, 1.01, 0.05), color=GRAY, label="all")
    ax.hist(u.auc_0_150[u.selective], bins=np.arange(0, 1.01, 0.05), color=BLUE, label="selective")
    ax.set_title(f"{area} ({100 * u.selective.mean():.1f}% selective)")
    ax.set_xlabel("AUC tHit vs tCR, 0-150 ms")
axes[0].set_ylabel("neurons")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()
print("Mean AUC range over neurons:", units.auc_0_150.min().round(3), "to", units.auc_0_150.max().round(3), "(all inside the axes)")

**Result: partially reproduced.** S1 30.5%, S2 16.7% and ALM 23.0% are within tolerance; MM is 36.4%, above 32.9 ± 3.29 and above the stated range widened by 10% (up to 36.3%); over four other seeds MM gives 35.0 to 36.4%. 165 of 191 selective neurons (86.4%) have mean AUC above 0.5.

**Result: partially reproduced.** S2 2.5%, MM 20.0% and ALM 9.8% are within tolerance, S1 is 5.6% (10 neurons) against 4.5% (3.4 to 5.6% over four other seeds). MM and ALM each exceed S1 and S2.

**Result: reproduced.** Before the stimulus 0% of S1, S2 and MM neurons and 0.8% of ALM neurons discriminate tactile from visual trials, as in the paper.

**Result: reproduced.** After the stimulus S1 36.2%, S2 35.8%, MM 13.6% and ALM 14.5% of neurons discriminate the stimulus, all within tolerance of 37.3, 35.2, 15 and 14.1%.

In [ ]:
rows = []
for area in xm.AREAS:
    u = units[(units.area == area) & units.rule_sig]
    r, p = stats.pearsonr(u.rule_auc, u.auc_0_50) if len(u) > 2 else (np.nan, np.nan)
    rho, p_s = stats.spearmanr(u.rule_auc, u.auc_0_50) if len(u) > 2 else (np.nan, np.nan)
    ua = units[units.area == area]
    r_all, p_all = stats.pearsonr(ua.rule_auc, ua.auc_0_50)
    rows.append({"area": area, "neurons": len(u), "pearson_r": r, "p": p, "spearman_rho": rho, "p_spearman": p_s,
                 "r_all_neurons": r_all, "p_all": p_all,
                 "x_range": f"{u.rule_auc.min():.2f}-{u.rule_auc.max():.2f}" if len(u) else "",
                 "y_range": f"{u.auc_0_50.min():.2f}-{u.auc_0_50.max():.2f}" if len(u) else ""})
f2f = pd.DataFrame(rows)
f2f["paper_r"] = [0.69, 0.91, 0.93, 0.83]
f2f["paper_neurons"] = [8, 4, 30, 26]
print(f2f.round(4).to_string(index=False))

**Result: reproduced.** Among rule-discriminating neurons, r = 0.958 in MM (28 neurons, p < 0.001) and 0.841 in ALM (25 neurons, p < 0.001), within tolerance of 0.93 and 0.83. S1 gives r = 0.681 (10 neurons, p = 0.030) and S2 0.902 (4 neurons, p = 0.0985).

## fig3b-rule-decoding

> The rule can be decoded from pre-stimulus population activity; median LDA accuracy, true against
> shuffled, S1 0.61/0.5, S2 0.62/0.53, MM 0.7/0.52, ALM 0.68/0.55.

In [ ]:
rng_dec = np.random.default_rng(303)
rows = []
for s in main:
    sel, block, stim = xm.correct_trials(s["trials"])
    for name, y, b in (("rule", block, 0), ("stimulus_pre", stim, 0), ("stimulus_post", stim, 1)):
        x = s["r100"][sel][:, :, b]
        rows.append({"area": s["area"], "subject": s["subject"], "units": s["n_units"], "what": name,
                     "true": xm.cv_accuracy(x, y, rng_dec), "shuffled": xm.cv_accuracy(x, rng_dec.permutation(y), rng_dec)})
dec = pd.DataFrame(rows)
med = dec.groupby(["what", "area"], sort=False)[["true", "shuffled"]].median().round(3)
print("Median accuracy over sessions:")
print(med.to_string())
r = dec[dec.what == "rule"]
print("\nRule decoding per session: range", r.true.min().round(3), "to", r.true.max().round(3),
      "; sessions above their shuffled value:", int((r.true > r.shuffled).sum()), "of", len(r))
print(r.groupby("area", sort=False).apply(lambda g: pd.Series({"sessions": len(g), "min": g.true.min(), "max": g.true.max(),
                                                              "wilcoxon_p_vs_shuffled": stats.wilcoxon(g.true, g.shuffled).pvalue})).round(3).to_string())
print("\nBy mouse (median rule accuracy):")
print(r.groupby(["area", "subject"], sort=False).true.median().round(3).to_string())

**Result: reproduced.** Median rule-decoding accuracy is S1 0.606, S2 0.617, MM 0.708 and ALM 0.682, all within 10% of the paper, against shuffled medians of 0.564, 0.532, 0.526 and 0.535. 37 of 40 sessions exceed their shuffled value.

## fig3c-rule-decoding-motor-stronger and fig3de-stimulus-decoding-controls

> Rule decoding CIs do not overlap shuffled ones in MM and ALM and overlap in S1 and S2; the stimulus is
> decoded after, not before, its onset.

In [ ]:
rng_boot = np.random.default_rng(304)
rows = []
for name, target, b in (("rule", "block", 0), ("stimulus_pre", "stimulus", 0), ("stimulus_post", "stimulus", 1)):
    for area in xm.AREAS:
        bb = xm.bootstrap_decoding([s for s in main if s["area"] == area], target, b, 200, rng_boot)
        lo, hi = np.quantile(bb, [0.025, 0.975], axis=0)
        rows.append({"what": name, "area": area, "true_mean": bb[:, 0].mean(), "true_lo": lo[0], "true_hi": hi[0],
                     "shuffled_mean": bb[:, 1].mean(), "shuffled_lo": lo[1], "shuffled_hi": hi[1],
                     "overlap": bool(lo[0] <= hi[1] and lo[1] <= hi[0])})
boot_dec = pd.DataFrame(rows)
print(boot_dec.round(3).to_string(index=False))
print("\nPaper, rule: S1 [0.52,0.61] vs [0.49,0.55]; S2 [0.53,0.63] vs [0.49,0.54]; MM [0.60,0.73] vs [0.48,0.53]; ALM [0.56,0.66] vs [0.49,0.54]")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 2.8), sharey=True)
for ax, name in zip(axes, ("rule", "stimulus_pre", "stimulus_post")):
    d = boot_dec[boot_dec.what == name].reset_index(drop=True)
    x = np.arange(len(d))
    ax.errorbar(x - 0.1, d.true_mean, yerr=[d.true_mean - d.true_lo, d.true_hi - d.true_mean], fmt="o", color=BLUE, label="true")
    ax.errorbar(x + 0.1, d.shuffled_mean, yerr=[d.shuffled_mean - d.shuffled_lo, d.shuffled_hi - d.shuffled_mean],
                fmt="o", color=GRAY, label="shuffled")
    ax.axhline(0.5, color="#cccccc", lw=1, zorder=0)
    ax.set_xticks(x, d.area)
    ax.set_title(name.replace("_", " "))
axes[0].set_ylabel("accuracy, 95% CI")
axes[0].legend(frameon=False)
plt.tight_layout()
plt.show()
print("Plotted range:", boot_dec[["true_lo", "shuffled_lo"]].min().min().round(3), "to",
      boot_dec[["true_hi", "shuffled_hi"]].max().max().round(3))

**Result: reproduced.** The bootstrap CIs of true and shuffled accuracy do not overlap in MM (0.601 to 0.731 against 0.472 to 0.523) and ALM (0.560 to 0.655 against 0.495 to 0.544) and overlap in S1 (0.530 to 0.614 against 0.493 to 0.545) and S2 (0.528 to 0.619 against 0.488 to 0.539), close to the paper's intervals.

**Result: reproduced.** Before the stimulus the true and shuffled CIs overlap in all four areas (true means 0.495 to 0.509); after it they do not overlap in any (true means 0.642 to 0.805).

## figs3-svm-rf-decoding

> SVM and random forests decode the rule about as well as LDA (medians in the claims file).

In [ ]:
rng_svm = np.random.default_rng(305)

class LinearSVM:
    def fit(self, x, y):
        self.model = SVC(kernel="linear", C=1.0).fit(x, y)
        return self

    def predict(self, x):
        return self.model.predict(x)

rows = []
for s in main:
    t = s["trials"]
    sel, block, _ = xm.correct_trials(t)
    x = s["r100"][sel][:, :, 0]
    ref = s["r100"][(t.category != "").to_numpy()][:, :, 0]
    xs = xm.soft_normalize(x.T, ref.T).T
    xz = (x - ref.mean(0)) / (ref.std(0, ddof=1) + 0.01)
    yp = rng_svm.permutation(block)
    row = {"area": s["area"], "units": s["n_units"],
           "svm": xm.cv_accuracy(xs, block, rng_svm, make_model=LinearSVM),
           "svm_shuffled": xm.cv_accuracy(xs, yp, rng_svm, make_model=LinearSVM),
           "svm_zscore": xm.cv_accuracy(xz, block, rng_svm, make_model=LinearSVM),
           "svm_zscore_shuffled": xm.cv_accuracy(xz, yp, rng_svm, make_model=LinearSVM)}
    for name, y in (("rf", block), ("rf_shuffled", yp)):
        forest = RandomForestClassifier(500, max_features="sqrt", oob_score=True, n_jobs=4,
                                        random_state=int(rng_svm.integers(2**31))).fit(xs, y)
        row[name] = forest.oob_score_
    rows.append(row)
svm = pd.DataFrame(rows)
med = svm.groupby("area").median().loc[xm.AREAS].drop(columns="units")
med["svm_paper"] = [0.60, 0.61, 0.71, 0.65]
med["rf_paper"] = [0.59, 0.60, 0.65, 0.70]
print(med.round(3).to_string())
within = sum(abs(med.loc[a, m] - med.loc[a, m + "_paper"]) <= 0.1 * med.loc[a, m + "_paper"] for a in xm.AREAS for m in ("svm", "rf"))
print("\nTrue medians within 10% of the paper:", within, "of 8; all above shuffled:",
      bool(((med.svm > med.svm_shuffled) & (med.rf > med.rf_shuffled)).all()))
print("Per-session range: SVM", svm.svm.min().round(3), "-", svm.svm.max().round(3), "; RF", svm.rf.min().round(3), "-", svm.rf.max().round(3))

**Result: reproduced.** All eight true medians are within 10% of the paper (SVM 0.606, 0.589, 0.658, 0.648; random forest 0.586, 0.606, 0.676, 0.655 for S1, S2, MM, ALM) and all exceed their shuffled medians.

## fig4c-prestim-distance-predicts-divergence

> Sessions with a larger pre-stimulus tHit-tCR distance show a larger distance after the stimulus
> (permutation test, p < 0.001, 40 sessions).

In [ ]:
rng_dist = np.random.default_rng(404)

def split_test(dist, n_perm=1000):
    pre = dist[:, :10].mean(1)
    top = pre >= np.median(pre)
    post = dist[:, 10:25]
    stat = np.linalg.norm(post[top].mean(0) - post[~top].mean(0))
    null = np.array([np.linalg.norm(post[p].mean(0) - post[~p].mean(0))
                     for p in (rng_dist.permutation(len(pre)) < top.sum() for _ in range(n_perm))])
    return top, post[top].mean(), post[~top].mean(), stat, float(np.mean(null >= stat))

dist = np.array([xm.trajectory_distance(s) for s in main])
dist_code = np.array([xm.trajectory_distance(s, offset=0.0) for s in main])
for name, d in (("Methods normalization (range + 5)", dist), ("code normalization (range)", dist_code)):
    top, a, b, stat, p = split_test(d)
    print(f"{name}: top {top.sum()} sessions mean post distance {a:.3f}, bottom {(~top).sum()} {b:.3f}; "
          f"statistic {stat:.3f}, permutation p = {p:.3f}")
    areas = pd.Series([s["area"] for s in main])
    print("   top-group sessions by area:", areas[top].value_counts().to_dict())
print("Distance range over sessions and bins:", dist.min().round(3), "to", dist.max().round(3))
pre, post = dist[:, :10].mean(1), dist[:, 10:25].mean(1)
print("Spearman correlation of pre and post mean distance over 40 sessions:", stats.spearmanr(pre, post))

**Result: partially reproduced.** The top 20 sessions by pre-stimulus distance have a larger mean post-stimulus distance than the bottom 20 (1.171 against 0.966), with permutation p = 0.004, not below 0.001. The code's normalization gives p = 0.001.

## fig4e-subspace-overlap and fig4f-prestim-overlap-predicts-poststim

> tCR overlap minus control tHit overlap: S1 -0.23, S2 -0.23, MM -0.36, ALM -0.35 (paired t-tests); pre- and
> post-stimulus tCR overlaps correlate, r = 0.68, 31 sessions.

In [ ]:
rng_ov = np.random.default_rng(505)
rows = []
for s in main:
    o = xm.subspace_overlaps(s, rng_ov, n_splits=100)
    single = [xm.subspace_overlaps(s, rng_ov, n_splits=1, ref_offset=5.0, cmp_offset=0.01, per_neuron_split=True)
              for _ in range(200)]
    rows.append({"area": s["area"], "units": s["n_units"], **o,
                 "code_control": [v["control_post"] for v in single], "code_cr": [v["cr_post"] for v in single],
                 "code_cr_pre": [v["cr_pre"] for v in single]})
ov = pd.DataFrame(rows)

def overlap_tests(df, ctrl="control_post", cr="cr_post"):
    out = []
    for area in xm.AREAS:
        g = df[df.area == area]
        d = g[cr] - g[ctrl]
        out.append({"area": area, "sessions": len(g), "difference": d.mean(), "p_paired_t": stats.ttest_rel(g[cr], g[ctrl]).pvalue,
                    "sessions_negative": int((d < 0).sum())})
    return pd.DataFrame(out)

main_ov = overlap_tests(ov[ov.units >= 10])
main_ov["paper"] = [-0.23, -0.23, -0.36, -0.35]
main_ov["within_10pct"] = (main_ov.difference - main_ov.paper).abs() <= 0.1 * main_ov.paper.abs()
print("Main (at least 10 units, 100 equal splits, offset 5):")
print(main_ov.round(4).to_string(index=False))
print("\nVariant, code's session rule (more than 10 units):")
print(overlap_tests(ov[ov.units > 10]).round(4).to_string(index=False))
# The code's single per-neuron split with offsets 5 and 0.01, repeated 200 times: distribution of each area's mean difference.
code_runs = []
g10 = ov[ov.units > 10]
for k in range(200):
    for area in xm.AREAS:
        g = g10[g10.area == area]
        code_runs.append({"run": k, "area": area,
                          "difference": np.mean([c[k] - h[k] for c, h in zip(g.code_cr, g.code_control)]),
                          "p": stats.ttest_rel([c[k] for c in g.code_cr], [h[k] for h in g.code_control]).pvalue})
code_runs = pd.DataFrame(code_runs)
print("\nVariant, code's procedure (one split per run, 200 runs): difference 2.5, 50, 97.5 percentiles, and runs with p < 0.05")
print(code_runs.groupby("area", sort=False).agg(p2_5=("difference", lambda d: d.quantile(0.025)), median=("difference", "median"),
                                                 p97_5=("difference", lambda d: d.quantile(0.975)),
                                                 runs_p_below_0_05=("p", lambda p: int((p < 0.05).sum()))).round(3).to_string())
print("Overlap range:", ov[["control_post", "cr_post", "cr_pre"]].min().min().round(3), "to", ov[["control_post", "cr_post", "cr_pre"]].max().max().round(3))

**Result: partially reproduced.** The tCR overlap is lower than the control tHit overlap in all four areas (S1 -0.291, p = 0.0007; S2 -0.218, p = 0.025; MM -0.466, p = 0.003; ALM -0.422, p < 0.0001), but only S2 is within 10% of the paper's difference; MM and ALM are larger in magnitude. The code's single-split procedure gives medians of -0.286, -0.222, -0.445 and -0.420 over 200 runs, and its 95% ranges exclude the paper's MM (-0.36) and ALM (-0.35) values.

In [ ]:
for label, g in (("at least 10 units (main)", ov[ov.units >= 10]), ("more than 10 units (code)", ov[ov.units > 10])):
    r, p = stats.pearsonr(g.cr_pre, g.cr_post)
    print(f"{label}: {len(g)} sessions, Pearson r = {r:.3f}, p = {p:.2g}")
    for area in xm.AREAS:
        a = g[g.area == area]
        print(f"   {area}: {len(a)} sessions, r = {stats.pearsonr(a.cr_pre, a.cr_post)[0]:.2f}")
g = ov[ov.units > 10]
rs = [stats.pearsonr([c[k] for c in g.code_cr_pre], [c[k] for c in g.code_cr])[0] for k in range(200)]
print("Code's single-split procedure, 200 runs: r 2.5, 50, 97.5 percentiles", np.quantile(rs, [0.025, 0.5, 0.975]).round(3))

**Result: reproduced.** Over 32 sessions with at least 10 units, r = 0.688 (p = 1.3e-05); with the code's 31 sessions r = 0.686. The paper gives 0.68.

## figs4ef-coding-dimensions and figs4h-choice-right-left

> Stimulus CDs are aligned between rules in S1, S2 and MM but not ALM; choice CDs are not aligned in any
> area. Right-against-left choice CDs are still not well aligned.

For figs4h the criterion's "needed condition" was first read as each of the four rare categories (left
licks in touch blocks, right licks in light blocks, by stimulus); that leaves 1 of 10 S1 sessions, so the
criterion cannot decide. As the code averages over the stimulus types that exist (`nanmean`), the right-
and left-lick trials are pooled over stimuli and need at least 2 each; the first reading is shown as a
variant.

In [ ]:
rng_cd = np.random.default_rng(606)
cd_rows, cd_boot = [], {}
for area in xm.AREAS:
    ss = [s for s in main if s["area"] == area]
    variants = {"main": (ss, "pooled"), "each_category": (ss, "each")}
    if any(s["n_units"] <= 10 for s in ss):
        variants["more_than_10_units"] = ([s for s in ss if s["n_units"] > 10], "pooled")
    for vname, (sub, rule) in variants.items():
        df = xm.bootstrap_coding_dimensions(sub, 1000 if vname == "main" else 300, rng_cd, rl_rule=rule)
        cd_boot[(area, vname)] = df
        for k in ("stim", "choice", "rl"):
            if vname == "each_category" and k != "rl":
                continue
            tq, sq = df[k + "_true"].quantile([0.025, 0.975]), df[k + "_shuffled"].quantile([0.025, 0.975])
            cd_rows.append({"area": area, "variant": vname, "sessions": len(sub), "cd": k,
                            "true": df[k + "_true"].mean(), "true_lo": tq.iloc[0], "true_hi": tq.iloc[1],
                            "shuffled": df[k + "_shuffled"].mean(), "shuffled_lo": sq.iloc[0], "shuffled_hi": sq.iloc[1],
                            "aligned": bool(tq.iloc[0] > sq.iloc[1] or sq.iloc[0] > tq.iloc[1]),
                            "resamples_with_value": int(df[k + "_true"].notna().sum())})
cd = pd.DataFrame(cd_rows)
print(cd.round(3).to_string(index=False))
rl_sessions = {area: sum(min(sum(xm.is_category(s["trials"], (b, st, d)).sum() for st in (1, 2)) for b in (1, 2) for d in (1, 2)) >= 2
                         for s in main if s["area"] == area) for area in xm.AREAS}
print("\nSessions with at least 2 right- and 2 left-lick trials in each block (pooled over stimuli):", rl_sessions)

In [ ]:
m = cd[cd.variant == "main"].set_index(["cd", "area"])
expected_aligned = {("stim", "S1"): True, ("stim", "S2"): True, ("stim", "MM"): True, ("stim", "ALM"): False,
                    **{("choice", a): False for a in xm.AREAS}}
match = {k: m.loc[k, "aligned"] == v for k, v in expected_aligned.items()}
print("figs4ef judgements matching the paper:", sum(match.values()), "of 8;", {f"{k[0]} {k[1]}": bool(v) for k, v in match.items()})
paper_rl = {"S1": 0.39, "S2": 0.32, "MM": 0.35, "ALM": 0.28}
for a in xm.AREAS:
    v = m.loc[("rl", a), "true"]
    print(f"figs4h {a}: true mean {v:.3f} (paper {paper_rl[a]}), below 0.5: {v < 0.5}, within 0.1: {abs(v - paper_rl[a]) <= 0.1}")

**Result: partially reproduced.** Seven of eight judgements match: stimulus CDs are aligned in S1 (0.557 against shuffled up to 0.384) and S2 (0.620 against up to 0.361) and not in ALM, and choice CDs are not aligned in any area, but the MM stimulus CD is not aligned (0.366, CI 0.218 to 0.521, against shuffled 0.150 to 0.441), where the paper reports 0.48 against 0.24.

**Result: partially reproduced.** Right-against-left choice CDs have true means below 0.5 in all four areas (S1 0.438, S2 0.422, MM 0.445, ALM 0.307), and S1, MM and ALM are within 0.1 of the paper; S2 is 0.10 above it (0.32). The CIs overlap the shuffled ones in all four areas.

## fig5b-first-hit-switches

> 88 switches into touch blocks and 91 into light blocks; mice switched before or immediately after the
> cue.

In [ ]:
blocks = []
for s in main:
    _, b = xm.transition_info(s["trials"])
    blocks.append(b.assign(session=s["path"], area=s["area"]))
blocks = pd.concat(blocks, ignore_index=True)
nonfirst = blocks[~blocks.first_block]
with_hit = nonfirst[nonfirst.first_hit.notna()]
print("Blocks of each type, all:", blocks.groupby("block").size().to_dict(),
      "; first blocks of a session:", blocks[blocks.first_block].groupby("block").size().to_dict())
print("Switches (non-first blocks) with a hit:", with_hit.groupby("block").size().to_dict(), "(paper: touch 88, light 91)")
c = with_hit[with_hit.cue.notna()].copy()
c["rel"] = c.first_hit.astype(float) - c.cue.astype(float)
c["at_or_before_cue_plus_1"] = c.rel <= 1
print("First hit relative to the cue row (rows of the file, removed trials not counted):")
print(c.groupby("block").agg(blocks_with_cue=("rel", "size"), fraction_by_cue_plus_1=("at_or_before_cue_plus_1", "mean"),
                             median_rel=("rel", "median"), min_rel=("rel", "min"), max_rel=("rel", "max")).round(3).to_string())
print("Blocks with a hit but no cue row in the file:", int(with_hit.cue.isna().sum()))
print("Trials from block start to first hit (rows): median", (with_hit.first_hit.astype(float) - with_hit.start).median() + 1)

**Result: partially reproduced.** Blocks after the first of a session that contain a hit number 63 (touch) and 75 (light), outside tolerance of 88 and 91; the files hold 88 touch and 91 light blocks when the first block of each session is counted. More than half of the first hits come before the cue trial or on the trial after it (touch 0.571, light 0.652).

## fig5de-motor-states-track-switch and figs5-sensory-states-no-tracking

> Kendall's tau of the touch-block probability over transitions: touch to light MM -0.39 [-0.67, -0.11],
> ALM -0.29 [-0.54, -0.04]; light to touch MM 0.37 [0.07, 0.63], ALM 0.27 [0.03, 0.49]; S1 and S2 CIs include 0.

In [ ]:
rng_tr = np.random.default_rng(707)
true_tau = pd.DataFrame([{"area": s["area"], **{k: v for k, v in xm.transition_taus(s, rng_tr).items() if not k.endswith("values")}}
                         for s in main])
print("Mean tau over sessions, 10-fold estimate (no resampling):")
print(true_tau.groupby("area", sort=False)[["to_light", "to_touch", "pre_touch", "pre_light"]].mean().round(3).to_string())
rows = []
for area in xm.AREAS:
    ss = [s for s in main if s["area"] == area]
    boot = []
    for b in range(500):
        vals = [xm.transition_taus(ss[j], rng_tr, bootstrap=True) for j in rng_tr.integers(len(ss), size=len(ss))]
        boot.append((np.nanmean([v["to_light"] for v in vals]), np.nanmean([v["to_touch"] for v in vals])))
    boot = np.array(boot)
    for i, name in enumerate(("to_light", "to_touch")):
        lo, hi = np.nanquantile(boot[:, i], [0.025, 0.975])
        rows.append({"area": area, "transition": name, "sessions": len(ss), "mean": np.nanmean(boot[:, i]), "lo": lo, "hi": hi,
                     "excludes_0": bool(lo > 0 or hi < 0)})
taus = pd.DataFrame(rows)
taus["paper"] = ["-0.16 [-0.43, 0.1]", "0.21 [-0.07, 0.5]", "-0.15 [-0.54, 0.21]", "0.25 [-0.17, 0.63]",
                 "-0.39 [-0.67, -0.11]", "0.37 [0.07, 0.63]", "-0.29 [-0.54, -0.04]", "0.27 [0.03, 0.49]"]
print("\nBootstrap (500 resamples of sessions, neurons and trials):")
print(taus.round(3).to_string(index=False))

In [ ]:
n_trans = []
for s in main:
    f, _ = xm.transition_info(s["trials"])
    n_trans.append({"area": s["area"], "early": int(f.early.sum()), "late": int(f.late.sum()), "train": int(f.not_trans.sum())})
print("Transition trials per session (median, min):")
print(pd.DataFrame(n_trans).groupby("area", sort=False).agg(["median", "min"]).to_string())

**Result: reproduced.** All four CIs exclude 0 with the paper's sign: touch to light MM -0.401 [-0.667, -0.111] and ALM -0.302 [-0.538, -0.051]; light to touch MM 0.382 [0.074, 0.630] and ALM 0.268 [0.026, 0.513], close to the paper's values.

**Result: reproduced.** All four S1 and S2 CIs include 0 (S1 -0.163 [-0.433, 0.117] and 0.219 [-0.067, 0.500]; S2 -0.140 [-0.500, 0.250] and 0.259 [-0.105, 0.620]).

## Optogenetic inhibition (fig6c, figs6b, fig6d, figs7gh, fig6-ppc, fig6fg, fig6ij)

Changes are with laser minus without, per session, averaged within mouse and then over mice; 95% CIs from
10000 hierarchical resamples. "methods" resamples hit and false-alarm trials; "code" keeps the observed
false-alarm rate, as the code does.

In [ ]:
rng_opto = np.random.default_rng(808)

def counts_for(sub, key):
    return (np.array([len(L[key]) for L in sub.licks]), np.array([int(np.sum(L[key])) for L in sub.licks]))

def change(sub, task, measure, o):
    if task == "cross-modal":
        hit, fa = {"tactile": ((1, 1, 1), (1, 2, 1)), "visual": ((2, 2, 2), (2, 1, 2)), "tactile_hit": ((1, 1, 1), None)}[measure]
        c = {"h0": counts_for(sub, hit + (1,)), "ho": counts_for(sub, hit + (o,))}
        if fa:
            c.update({"f0": counts_for(sub, fa + (1,)), "fo": counts_for(sub, fa + (o,))})
    else:
        c = {"h0": counts_for(sub, (1, 1)), "ho": counts_for(sub, (1, o)), "f0": counts_for(sub, (2, 1)), "fo": counts_for(sub, (2, o))}
    formula = (lambda p: (p["ho"] - p["fo"]) - (p["h0"] - p["f0"])) if "f0" in c else (lambda p: p["ho"] - p["h0"])
    mouse = sub.mouse.to_numpy()
    b = xm.hierarchical_bootstrap(mouse, c, formula, 10000, rng_opto)
    bc = xm.hierarchical_bootstrap(mouse, c, formula, 10000, rng_opto, resample={"h0", "ho"})
    return {"point": xm.point_estimate(mouse, c, formula), "boot_mean": b.mean(),
            "lo": np.quantile(b, 0.025), "hi": np.quantile(b, 0.975),
            "code_lo": np.quantile(bc, 0.025), "code_hi": np.quantile(bc, 0.975),
            "sessions": len(sub), "mice": sub.mouse.nunique(), "trials_with_laser": int(c["ho"][0].sum())}

rows = []
for task, measures in (("cross-modal", ("tactile", "tactile_hit", "visual")), ("detection", ("sensitivity",))):
    for measure in measures:
        for o, window in ((2, "pre"), (3, "post")):
            for cond in ("sham", "S1/S2", "MM", "ALM", "AMM", "PPC"):
                sub = optos[(optos.task == task) & (optos.condition == cond)]
                rows.append({"task": task, "measure": measure, "window": window, "condition": cond, **change(sub, task, measure, o)})
opto_res = pd.DataFrame(rows)
opto_res["ci_excludes_0"] = np.where(opto_res.hi < 0, "below", np.where(opto_res.lo > 0, "above", "includes 0"))
opto_res["code_ci_excludes_0"] = np.where(opto_res.code_hi < 0, "below", np.where(opto_res.code_lo > 0, "above", "includes 0"))
print(opto_res.round(3).to_string(index=False))

In [ ]:
def show(task, measure, window, conds, paper):
    d = opto_res[(opto_res.task == task) & (opto_res.measure == measure) & (opto_res.window == window)].set_index("condition").loc[conds]
    d = d[["sessions", "mice", "point", "lo", "hi", "ci_excludes_0", "code_lo", "code_hi", "code_ci_excludes_0"]].copy()
    d["paper_ci"] = paper
    print(f"{task}, {measure}, laser {window}-stimulus")
    print(d.round(3).to_string())
    print()

show("cross-modal", "tactile", "pre", ["S1/S2", "MM", "ALM", "sham", "AMM"],
     ["[-0.47, 0.08]", "[-0.65, -0.12]", "[-0.68, -0.02]", "[-0.12, 0.13]", "[-0.19, 0.06]"])

**Result: partially reproduced.** Pre-stimulus inhibition of MM lowers tactile sensitivity (-0.390, CI -0.658 to -0.123), and S1/S2, sham and AMM CIs include 0; for ALM the change is -0.373 but its CI reaches 0.005 when false alarms are also resampled. With the code's resampling the ALM CI is -0.682 to -0.018, as in the paper (-0.68 to -0.02).

In [ ]:
show("cross-modal", "tactile_hit", "pre", ["S1/S2", "MM", "ALM"], ["[-0.31, 0.01]", "[-0.49, -0.15]", "[-0.46, -0.07]"])

**Result: reproduced.** The tactile hit rate falls with MM (-0.318, CI -0.492 to -0.152) and ALM (-0.227, -0.463 to -0.007) inhibition, and the S1/S2 CI includes 0 (-0.303 to 0.010).

In [ ]:
show("cross-modal", "tactile", "post", ["S1/S2", "MM", "ALM"], ["[-0.67, -0.08]", "[-0.6, -0.34]", "[-0.55, -0.27]"])

**Result: reproduced.** Inhibition from the stimulus onset lowers tactile sensitivity for S1/S2 (-0.389, CI -0.699 to -0.080), MM (-0.487, -0.612 to -0.327) and ALM (-0.402, -0.568 to -0.253).

In [ ]:
show("cross-modal", "visual", "pre", ["MM", "AMM", "ALM", "S1/S2"], ["[-0.6, -0.18]", "[-0.35, -0.08]", "[-0.5, 0.04]", "[-0.27, 0.05]"])
show("cross-modal", "visual", "post", ["MM", "ALM", "AMM", "S1/S2"], ["[-0.83, -0.52]", "[-0.84, -0.39]", "[-0.72, -0.36]", "[-0.44, 0.04]"])

**Result: reproduced.** All eight judgements match: pre-stimulus MM (-0.603 to -0.173) and AMM (-0.356 to -0.082) CIs are below 0, ALM and S1/S2 include 0; post-stimulus MM, ALM and AMM are below 0 and S1/S2 includes 0 (-0.460 to 0.045).

In [ ]:
for measure, window, paper in (("tactile", "pre", "[-0.4, -0.06]"), ("tactile", "post", "[-0.43, -0.04]"),
                               ("visual", "pre", "[-0.31, -0.05]"), ("visual", "post", "[-0.54, -0.07]")):
    show("cross-modal", measure, window, ["PPC"], [paper])

**Result: reproduced.** PPC inhibition lowers tactile sensitivity before (-0.408 to -0.052) and after (-0.466 to -0.009) the stimulus onset and visual sensitivity before (-0.333 to -0.031) and after (-0.555 to -0.070).

In [ ]:
show("detection", "sensitivity", "pre", ["S1/S2", "MM", "ALM", "AMM", "PPC", "sham"],
     ["[-0.34, 0.18]", "[-0.23, 0.06]", "[-0.28, 0.07]", "[-0.25, 0.16]", "[-0.12, 0.1]", ""])
show("detection", "sensitivity", "post", ["S1/S2", "MM", "ALM", "AMM", "PPC", "sham"],
     ["[-0.59, -0.31]", "[-0.59, -0.25]", "[-0.45, -0.1]", "[-0.17, 0.18]", "[-0.14, 0.3]", ""])

**Result: reproduced.** In the detection task, pre-stimulus inhibition CIs include 0 for S1/S2, MM and ALM, and post-stimulus CIs are below 0 (S1/S2 -0.643 to -0.251, MM -0.628 to -0.233, ALM -0.471 to -0.065); AMM and PPC include 0 in both windows.

In [ ]:
rows = []
for task in ("cross-modal", "detection"):
    for duration in ("short", "long"):
        for cond in ("sham", "S1/S2", "MM", "ALM", "AMM", "PPC"):
            sub = optos[(optos.task == task) & (optos.condition == cond)]
            if task == "cross-modal":
                iti, rw = ((1, 3, 1, 1), (1, 3, 1, 2)) if duration == "short" else ((1, 4, 1, 1), (1, 4, 1, 3))
            else:
                iti, rw = ((4, 1), (4, 3)) if duration == "short" else ((5, 1), (5, 3))
            c = {"iti": counts_for(sub, iti), "rw": counts_for(sub, rw)}
            f = lambda p: p["rw"] - p["iti"]
            b = xm.hierarchical_bootstrap(sub.mouse.to_numpy(), c, f, 10000, rng_opto)
            lo, hi = np.quantile(b, [0.025, 0.975])
            rows.append({"task": task, "laser": duration, "condition": cond, "laser_only_trials": int(c["rw"][0].sum()),
                         "p_iti": c["iti"][1].sum() / max(c["iti"][0].sum(), 1), "p_response": c["rw"][1].sum() / max(c["rw"][0].sum(), 1),
                         "point": xm.point_estimate(sub.mouse.to_numpy(), c, f), "lo": lo, "hi": hi,
                         "includes_0": bool(lo <= 0 <= hi)})
laser_only = pd.DataFrame(rows)
print(laser_only.round(3).to_string(index=False))
key = laser_only[laser_only.condition.isin(["S1/S2", "MM", "ALM"])]
print("\nS1/S2, MM and ALM CIs including 0:", int(key.includes_0.sum()), "of", len(key))

**Result: not reproduced.** 11 of the 12 CIs include 0, but on short laser-only trials of the detection task MM inhibition raises the right-lick probability (0.258, CI 0.045 to 0.483). All six cross-modal CIs and all long-laser CIs include 0.

## Summary

| Claim | Outcome |
|---|---|
| fig1e-recording-counts | reproduced |
| fig6-session-counts | reproduced |
| fig1d-performance | reproduced |
| fig1d-outcome-fractions | reproduced |
| fig1h-thit-tcr-selective | partially reproduced |
| fig2c-rule-prestim-units | partially reproduced |
| fig2d-stimulus-prestim-units | reproduced |
| fig2e-stimulus-poststim-units | reproduced |
| fig2f-prestim-predicts-touch-response | reproduced |
| fig3b-rule-decoding | reproduced |
| fig3c-rule-decoding-motor-stronger | reproduced |
| fig3de-stimulus-decoding-controls | reproduced |
| figs3-svm-rf-decoding | reproduced |
| fig4c-prestim-distance-predicts-divergence | partially reproduced |
| fig4e-subspace-overlap | partially reproduced |
| fig4f-prestim-overlap-predicts-poststim | reproduced |
| figs4ef-coding-dimensions | partially reproduced |
| figs4h-choice-right-left | partially reproduced |
| fig5b-first-hit-switches | partially reproduced |
| fig5de-motor-states-track-switch | reproduced |
| figs5-sensory-states-no-tracking | reproduced |
| fig6c-pre-inhibition-tactile | partially reproduced |
| figs6b-pre-inhibition-hit-rate | reproduced |
| fig6d-post-inhibition-tactile | reproduced |
| figs7gh-visual-sensitivity | reproduced |
| fig6-ppc-inhibition | reproduced |
| fig6fg-detection-task | reproduced |
| fig6ij-laser-only-licks | not reproduced |
| fig1e-probe-locations | blocked (data missing) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the reasoning and
the differences from the paper's method.

**How the analysis differs from the paper's.** The analysis is in Python, not MATLAB. Bootstrap resamples
for AUC are shared by all units and bins of a session. The decoding CIs use 200 resamples per area, the
transition CIs 500 and the coding-dimension CIs 1000. The transition classifier is LDA with Ledoit-Wolf
shrinkage in place of the code's optimized LDA. Subspace overlaps average 100 equal splits of the
tactile hits; the code's single split per neuron is a variant. Where the Methods and code disagree
(normalization offsets, the session rule of the subspace analysis, resampling of false alarms in the
optogenetic CIs), the Methods' choice is the main result and the code's is shown. The first-hit trial
numbers are counted in rows of the files, which lack the removed trials.

**What was found about the dandiset.** Its counts of sessions, mice, units and optogenetic sessions
match the paper exactly, and most results reproduce closely, so the dandiset holds the data the paper
analysed. The ephys files hold only the trials and units the paper kept: trials with licks between -1 and
0.1 s of the stimulus and units failing the quality filter are absent, which the dandiset does not say.
Units carry no quality metrics, channel, depth or original cluster id; electrode coordinates are NaN and
locations "unknown", and no histology is included. In the behaviour files `start_time` equals the stimulus
onset and trials overlap; in the detection task, trials without a whisker stimulus are labelled
"Stim_Vis". Trial table columns have no descriptions, and codes such as `response` (1 right, 2 left,
0 none) and `blockType` ("Change" for the cue trial) are explained only by the paper's code.

In [ ]:
print(f"Notebook ran in {(time.time() - started) / 60:.1f} minutes")